# COMPLETE-LINK CLUSTERING

## Instance Creation

### Load DFs

In [ ]:
import pickle
pickle_folder = r"C:\Users\mkoseogl\Desktop\Geneva Data\pickle_files"

with open(f"{pickle_folder}/main_datasets.pkl", "rb") as f:
    data = pickle.load(f)

unique_bike_stations = data["unique_bike_stations"]
unique_general_only = data["unique_general_only"]
bike_trips_clean = data["bike_trips_clean"]

print("✅ Pickle file loaded successfully!")

with open(f"{pickle_folder}/matched_bike_trips_bike_stations.pkl", "rb") as f:
    data = pickle.load(f)
                
matched_bike_trips_bike_stations = data['matched_bike_trips_bike_stations']

print("✅ Pickle file loaded successfully!")

display(matched_bike_trips_bike_stations.head(10))


### Clustering 

In [ ]:
from Codes.map_and_hub_creator_complete_link import create_hub_map_and_dataframe_complete_link
import pickle

m, hub_df_complete_link, stops_4326, polys_4326 = create_hub_map_and_dataframe_complete_link(unique_bike_stations, unique_general_only, threshold = 300, buffer = 100)

# Save each DataFrame to a separate pickle file
hub_df_complete_link.to_pickle(f"{pickle_folder}/hub_df_complete_link.pkl")
stops_4326.to_pickle(f"{pickle_folder}/stops_4326_complete_link.pkl")
polys_4326.to_pickle(f"{pickle_folder}/polys_4326_complete_link.pkl")

m.save("completelink_hub_map_sample.html")
print("Map saved as hub_map_sample.html")
m


### (OPTIONAL) Create Bike O-D Trip Counts For Demand Generation 

In [ ]:
from Codes.create_od_counts import build_bike_od_counts

od_counts = build_bike_od_counts(matched_bike_trips_bike_stations, hub_df_complete_link, pickle_path = f"{pickle_folder}/od_counts_complete_link.pkl")


### Sample the Network

In [ ]:
# This section creates samples of hubs based on specified radii around city center coordinates.

import geopandas as gpd
from shapely.geometry import Point
import pickle
'''
#load Pickle files from the path if necessary
with open(r"C:\Users\mkoseogl\Desktop\Geneva Data\pickle_files\hub_df_complete_link.pkl", "rb") as f:
    hub_df_complete_link = pickle.load(f)

with open(r"C:\Users\mkoseogl\Desktop\Geneva Data\pickle_files\od_counts_complete_link.pkl", "rb") as f:
    od_counts = pickle.load(f)
'''
# Define the city center coordinates by selecting hubs from the map 
hub_id_values = [781, 7, 4] #change
city_centers  = []
target_radii = [1500, 2500, 3500, 5000]  # Create a buffer (e.g., 3000 meters radius)

zones = []
hub_df_samples = []

for id in hub_id_values:
    for rad in target_radii:
            city_center = hub_df_complete_link.loc[hub_df_complete_link['hub_id'] == id, 'geometry'].iloc[0]
            city_centers.append(city_center)
            zone = city_center.buffer(rad)
            zones.append(zone)
            # Filter the hub_df to only include nodes within the zone
            hub_df_complete_link = gpd.GeoDataFrame(hub_df_complete_link, geometry='geometry', crs="EPSG:3857")
            hub_df_sample_run = hub_df_complete_link[hub_df_complete_link.geometry.within(zone)]
            hub_df_samples.append(hub_df_sample_run)
            print(f"Original number of hubs: {len(hub_df_complete_link)}")
            print(f"Number of hubs in target zone: {len(hub_df_sample_run)}")


### Run Instance Generator

In [ ]:
import random
import os
from Codes.CPPMTO_instance_generator_v2 import generate_instance_real

# Set a random seed for reproducibility (optional)
seed_number=42
random.seed(seed_number)

# Price Inputs
M = 4                               # assign upper bound for the price.

#if random choice parameters are needed. Modify Choice Parameters depending on the choice model used in Instance Generator

beta_cost_range = (0.8, 1)          # cost sensitivity
beta_time_range = (0.5, 0.75)       # time sensitivity
beta_transfer_range = (0.5, 0.75)   # transfer sensitivity
beta_buffer_range = (0.5, 0.75)     # for car trips


# Capacity constraints

capacity_range = (100, 150)           # capacity for each bike station
inventory_range = (30, 35)          # how many bikes actually available

# =====================================================
# 2. Create a folder to store the generated instances
# =====================================================

Data_folder=r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances" # Define the folder path to store the generated instances
folder_path = os.makedirs(Data_folder, exist_ok=True)
folder_path = Data_folder

json_paths = []

for hub in hub_df_samples:
    json_file_path, txt_file_path = generate_instance_real(folder_path, seed_number, hub, od_counts,
                           beta_cost_range, beta_time_range, beta_transfer_range, beta_buffer_range,
                           capacity_range, inventory_range, 
                           fixed_pt_price_value=4, bike_cost_operator=0.5, ridership_weight=2,
                           max_disp_alts=7, min_demand=10, max_demand=70, max_travel_time=80,
                           min_od_distance=500, bike_max_distance=3000, transfer_buffer_range=(1,10),
                           use_fixed_beta_values=True, 
                           fixed_beta_cost = 0.9, fixed_beta_time = 0.6, fixed_beta_transfer = 0.4, fixed_beta_buffer = 0.2                           
                           )
    json_paths.append(json_file_path)


### Transform Data for Balancing

In [ ]:
from Codes.instance_transformer import transform_bike_data
import glob

# De-randomizes the capacities and inventories of bike stations (OPTIONAL)

Data_folder = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\json"
json_paths = glob.glob(f"{Data_folder}/*.json")
pickle_folder = r"C:\Users\mkoseogl\Desktop\Geneva Data\pickle_files"

# Below function de-randomizes the capacities and inventories of bike stations in the JSON files created above. 
# It creates low inventory, high inventory and random inventory with capacity values of 20, 30 and 40. 


for path in json_paths:
    for base_value in [20, 30, 40]:
        transform_bike_data(path, df_path = f"{pickle_folder}/hub_df_complete_link.pkl", base_cap_value = base_value)


## Solve Bilevel Model

In [ ]:
import json, os, numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob

# Show all rows and columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)  # Allow automatic width
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.6f}".format)

#Solve Bilevel Model with Choice Parameters from Rojas et al. (2023)
    # Montes, A., Geržinic, N., Veeneman, W., van Oort, N., & Hoogendoorn, S. (2023). 
    # Shared micromobility and public transport integration - A mode choice study using stated preference data. 
    # Research in Transportation Economics, 99, Article 101302. https://doi.org/10.1016/j.retrec.2023.101302

from Codes.CPPMTO_optimizer1_resman_rojas_choice_params_2 import solve_bilevel_model_resman_rojas_2

from visualize_solutions.Bilevel_Solution_Visualization import summarize_experiment_solutions, Bilevel_Solution_Visualizer


resman_save_folder  = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\randrun"

path = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Data\json\randrun\updated_basecap_20_random_real_instance_42_32_762.json"

res_man = solve_bilevel_model_resman_rojas_2(path, resman_save_folder,  M=4, pt_price=4, theta = 0.5, verbose2=False)

summarize_experiment_solutions(resman_save_folder)


## Base Case Scenarios

Base case scenarios are single level problems that can be solved to see one sided optimal solutions. 

-LL Base Case only consider the utilities of the travelers and capacity constraints to receive output for a fixed price. 

-HPR Base Case do not consider user choices and focuses solely on operator decisions with lower level constraints. This gives an upper bound for the bilevel model objective value. This means the prices we see from this are going to be higher as there is no sensitivity to it. 


Donkey republic offers 2 CHF for the first 30 minutes of bike ride. 

In [ ]:
import json, os, numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob

# Show all rows and columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)  # Allow automatic width
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.6f}".format)

from Codes.CPPMTO_optimizer1_resman_basecase import solve_lower_level
from visualize_solutions.Bilevel_Solution_Visualization import summarize_experiment_solutions, Bilevel_Solution_Visualizer


base_case_save_folder  = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\randrun_base_case"
randrun_paths = glob.glob(r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Data\json\randrun\*.json")

for path in randrun_paths:
    res_man = solve_lower_level(path, base_case_save_folder,  bike_price= 2, pt_price=4, verbose=False)

from pathlib import Path

# --------------------------------------------------------------------
# 2.  Collect the summary rows
# --------------------------------------------------------------------
rows = []
for f in Path(base_case_save_folder).glob("*_LL_solution.json"):
    with open(f, "r") as fp:
        res = json.load(fp)

    # ----------------- robust extraction ----------------------------
    flow_totals = res.get("flow_totals", {})
    if not flow_totals and "flows" in res:           # older output format
        # rebuild quick totals
        pt_bike = sum(x["q"] for x in res["flows"] if x.get("type") == "PT+Bike")
        pt_only = sum(x["q"] for x in res["flows"] if x.get("type") == "PT")
        car     = sum(x["q"] for x in res["flows"] if x.get("type") == "Car")
        flow_totals = {"PT+Bike": pt_bike, "PT": pt_only, "Car": car}

    rows.append({
        "instance"      : f.stem,
        "solve_time_s"  : res.get("solution_time", None),   # only if you stored it
        "gap"           : res.get("gap", None),             # same – may be None
        "objective"     : res.get("objective"),
        "PT+Bike_flow"  : flow_totals.get("PT+Bike", 0),
        "PT_only_flow"  : flow_totals.get("PT", 0),
        "Car_flow"      : flow_totals.get("Car", 0),
        "revenue"       : res.get("revenue", 0)
    })

# --------------------------------------------------------------------
# 3.  Build and show the dataframe
# --------------------------------------------------------------------
df = pd.DataFrame(rows).set_index("instance")
display(df)     # if running in Jupyter; otherwise print(df)


### HPR Base Case

In [ ]:
import json, os, numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob

# Show all rows and columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)  # Allow automatic width
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.6f}".format)

from Codes.CPPMTO_optimizer1_resman_HPR_rojas_choice_params_2 import solve_hpr_rojas_2
from visualize_solutions.Bilevel_Solution_Visualization import summarize_experiment_solutions, Bilevel_Solution_Visualizer


hpr_save_folder  = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\randrun_hpr"
randrun_paths = glob.glob(r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Data\json\randrun\*.json")

for path in randrun_paths:
    res_man = solve_hpr_rojas_2(path, hpr_save_folder, P_bar=0.5, M=4, pt_price=4, theta = 0.5, verbose=False, verbose2=False)
hpr_save_folder  = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\randrun_hpr"

hpr_df = summarize_experiment_solutions(hpr_save_folder, show = True)


## Sensitivity Analysis

Sensitivity analysis on capacities, inventory levels and price lower bounds. Here we considered incentivization and profit/break-even scenario with different capacity/inventory scenarios. 

In [ ]:
import json, os, numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Show all rows and columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)  # Allow automatic width
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.6f}".format)

from Codes.CPPMTO_optimizer1_resman_rojas_choice_params_2 import solve_bilevel_model_resman_rojas_2

from pathlib import Path

folder_path = Path(r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Data\json\dems")
resman_save_folder = Path(r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems")

price_range = [0, 0.5]

for price_lb in price_range:
    print(f"\nRunning for price_lb = {price_lb}\n")

    # Recursively find all JSON files in folder_path
    json_paths = folder_path.rglob("*.json")

    for json_path in json_paths:
        # Derive relative subfolder like "20_low"
        relative_subfolder = json_path.parent.relative_to(folder_path)

        # Construct corresponding save path
        save_path = resman_save_folder / relative_subfolder
        save_path.mkdir(parents=True, exist_ok=True)

        print(f"Processing {json_path.name} into {save_path}")

        res_man = solve_bilevel_model_resman_rojas_2(
            json_path, 
            save_path, 
            P_bar=price_lb, 
            M=4, 
            pt_price=4, 
            theta=0.5, 
            verbose=False, 
            verbose2=False
        )

from visualize_solutions.Bilevel_Solution_Visualization import summarize_experiment_solutions, Bilevel_Solution_Visualizer

high_20= r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems\20_high"
low_20 = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems\20_low"
low_30 = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems\30_low"
high_30 = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems\30_high"
high_40 = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems\40_high"
low_40 = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems\40_low"

df_low_20 = summarize_experiment_solutions(low_20)
df_high_20 = summarize_experiment_solutions(high_20)
df_low_30 = summarize_experiment_solutions(low_30)
df_high_30 = summarize_experiment_solutions(high_30)
df_low_40 = summarize_experiment_solutions(low_40)
df_high_40 = summarize_experiment_solutions(high_40)



## Plots

Plotting the outputs. 

### Get Dataframes

In [ ]:
from visualize_solutions.Bilevel_Solution_Visualization import summarize_experiment_solutions, Bilevel_Solution_Visualizer
import pandas as pd
import glob

# Show all rows and columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)  # Allow automatic width
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.6f}".format)

resman_save_folder  = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\randrun"
base_case_save_folder  = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\randrun_base_case"
hpr_save_folder  = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\randrun_hpr"

high_20= r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems\20_high"
low_20 = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems\20_low"
low_30 = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems\30_low"
high_30 = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems\30_high"
high_40 = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems\40_high"
low_40 = r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\dems\40_low"


### Market Share Plots

In [ ]:
from Codes.market_share_plotter import plot_market_share_multi

runs= {"Base Case LL": base_case_df,
       "Base Case HPR": hpr_df,
       "Bilevel Model": resman_df
}

dataframes, _ = plot_market_share_multi(runs, title = "Market Share Comparison of Base Case and Bilevel Models")
display(dataframes["Base Case LL"])
display(dataframes["Base Case HPR"])
display(dataframes["Bilevel Model"])


## Solution Visualizer

Set path to the solution and data json folders. 
Select hub data pickle file to create the map.
Add individual pt/bike nodes (non clustered) if desired.
Add cluster polygons for visualizing the coverage area of the clusters.
Visualize the solutions through the O-D search engine.

In [ ]:
from visualize_solutions.Bilevel_Solution_Visualization import summarize_experiment_solutions, Bilevel_Solution_Visualizer

Bilevel_Solution_Visualizer(pickle_folder=r"C:\Users\mkoseogl\Desktop\Geneva Data\pickle_files",
                            network_folder=r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Data\json\randrun",
                            solution_folder=r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\complete_link_instances\Solutions\randrun"  )
